In [ ]:
from pathlib import Path
import os
import sys

try:
    import kaggle  # noqa: F401
except Exception:
    %pip -q install kaggle

PROJECT_ROOT = Path("/content/Sleep-Stage")
LOCAL_ROOT = Path("/Users/temicide/Documents/5_domain_final/Sleep-Stage")
if LOCAL_ROOT.exists():
    PROJECT_ROOT = LOCAL_ROOT

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Colab input path: /content/input
CONTENT_ROOT = Path("/content")
INPUT_ROOT = CONTENT_ROOT / "input"
WORKING_DIR = CONTENT_ROOT / "working"
SUBMISSION_PATH = CONTENT_ROOT / "submission.csv"
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using project root: {PROJECT_ROOT}")


In [ ]:
from pathlib import Path
import os

uploaded_kaggle_json = None
if not (Path.home() / ".kaggle" / "kaggle.json").exists():
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        from google.colab import files
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            uploaded_kaggle_json = Path("kaggle.json")
        else:
            raise RuntimeError("Upload kaggle.json, or set KAGGLE_USERNAME and KAGGLE_KEY in Colab secrets.")

from sleep_stage.kaggle_download import configure_kaggle_credentials

credential_path = configure_kaggle_credentials(uploaded_kaggle_json)
if credential_path is None:
    raise RuntimeError("Kaggle credentials are not configured.")
print("Kaggle credentials configured.")


In [ ]:
from sleep_stage.config import COMPETITION_SLUG
from sleep_stage.kaggle_download import download_and_extract_competition

# Downloads with: kaggle competitions download
competition_root = download_and_extract_competition(INPUT_ROOT, COMPETITION_SLUG)
print(f"Competition data ready: {competition_root}")


In [ ]:
from sleep_stage.config import build_paths
from sleep_stage.pipeline import load_or_build_features, run_experiments

paths = build_paths(
    project_root=PROJECT_ROOT,
    competition_root=competition_root,
    output_path=SUBMISSION_PATH,
    working_dir=WORKING_DIR,
)
train_table, test_table = load_or_build_features(paths, force=False)
print("train_table", train_table.shape)
print("test_table", test_table.shape)
print(train_table["label"].value_counts().to_dict())

results = run_experiments(paths=paths, force_features=False)
print(results[["experiment", "model_name", "n_features", "weighted_f1_mean", "weighted_f1_std"]].to_string(index=False))


In [ ]:
from pathlib import Path
import pandas as pd

from sleep_stage.data import validate_submission
from sleep_stage.pipeline import generate_submission

submission_path = generate_submission(paths)
validation = validate_submission(submission_path, paths.sample_submission)
submission = pd.read_csv(submission_path)

print(validation)
print(submission.head().to_string(index=False))
print(submission["labels"].value_counts().to_string())
print(f"Submission CSV ready: {submission_path}")
assert submission_path == Path("/content/submission.csv") or LOCAL_ROOT.exists()
assert len(submission) == 7832
